### Import Library & Load Data

In [32]:
import os
import joblib
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    mean_absolute_error,
)

pd.set_option('display.float_format', lambda x: '%.3f' % x)

# Memuat data hasil preprocessing yang disimpan dari tahap sebelumnya
data_prep = joblib.load('../dataset/data_hasil_preprocessing.pkl')

X_train = data_prep['X_train']
X_test = data_prep['X_test']
y_train = data_prep['y_train']  # Sudah dalam bentuk log1p(price)
y_test = data_prep['y_test']    # Sudah dalam bentuk log1p(price)
preprocessor = data_prep['preprocessor']

print("Data hasil preprocessing berhasil dimuat!")
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

Data hasil preprocessing berhasil dimuat!
X_train : (6058, 7)
X_test  : (1515, 7)
y_train : (6058,)
y_test  : (1515,)


### Fungsi Perhitungan Metrik Evaluasi

In [33]:
def hitung_metrik_jurnal(y_aktual, y_prediksi, jumlah_fitur):
    n = len(y_aktual)
    r2   = r2_score(y_aktual, y_prediksi)
    mae  = mean_absolute_error(y_aktual, y_prediksi)
    mse  = mean_squared_error(y_aktual, y_prediksi)
    rmse = np.sqrt(mse)
    
    adj_r2 = 1 - ((1 - r2) * (n - 1) / (n - jumlah_fitur - 1))
    
    return {
        'R2': round(r2, 4),
        'Mean absolute error': round(mae, 4),
        'Mean squared error': round(mse, 4),
        'Root mean squared error': round(rmse, 4),
        'Adjusted R2': round(adj_r2, 4)
    }

print("Fungsi perhitungan metrik evaluasi berhasil dibuat!")

Fungsi perhitungan metrik evaluasi berhasil dibuat!


### Hypertuning Parameter

In [34]:
# Inisialisasi Model Dasar XGBoost di dalam Pipeline 
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('xgb', XGBRegressor(random_state=42, n_jobs=-1))
])

# Skema 3-Fold Cross-Validation
kf = KFold(
    n_splits=3, 
    shuffle=True, 
    random_state=42
)

param_grid = {
    'xgb__n_estimators': [200, 300, 500],
    'xgb__max_depth': [3, 5, 7],
    'xgb__learning_rate': [0.01, 0.05, 0.1],
    'xgb__subsample': [0.8, 1.0],
    'xgb__colsample_bytree': [0.8, 1.0],
    'xgb__gamma': [0, 0.1, 0.2],
    'xgb__min_child_weight': [1, 3, 5]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=kf,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=2
)

print("Memulai GridSearchCV...")
grid_search.fit(X_train, y_train)

best_params = grid_search.best_params_

print("\nHYPERPARAMETER TERBAIK:")
for param, nilai in best_params.items():
    print(f"{param} : {nilai}")

print(f"\nBest Neg MSE Score: {grid_search.best_score_:.4f}")

Memulai GridSearchCV...
Fitting 3 folds for each of 972 candidates, totalling 2916 fits

HYPERPARAMETER TERBAIK:
xgb__colsample_bytree : 0.8
xgb__gamma : 0
xgb__learning_rate : 0.05
xgb__max_depth : 5
xgb__min_child_weight : 5
xgb__n_estimators : 500
xgb__subsample : 1.0

Best Neg MSE Score: -0.1455


### Pelatihan dan Pengujian Model XGBoost Final

In [35]:
# Mengambil estimator terbaik dari hasil GridSearchCV
best_model = grid_search.best_estimator_

# Melakukan prediksi pada data training (menggunakan cross_val_predict untuk evaluasi training yang adil)
y_pred_train_xgb = cross_val_predict(
    best_model,
    X_train,
    y_train,
    cv=kf,
    n_jobs=-1
)

# Melatih ulang model pada seluruh data training
best_model.fit(X_train, y_train)

# Melakukan prediksi pada data testing
y_pred_test_xgb = best_model.predict(X_test)

# Evaluasi model XGBoost menggunakan fungsi metrik
jumlah_fitur_encoded = best_model.named_steps['preprocessor'].transform(X_train).shape[1]

metrik_train_xgb = hitung_metrik_jurnal(y_train,y_pred_train_xgb,jumlah_fitur=jumlah_fitur_encoded)
metrik_test_xgb = hitung_metrik_jurnal(y_test,y_pred_test_xgb,jumlah_fitur=jumlah_fitur_encoded)


df_tabel_xgb = pd.DataFrame({
    f'Training ({len(X_train)})': metrik_train_xgb,
    f'Testing ({len(X_test)})': metrik_test_xgb
})

print("HASIL PREDIKSI MENGGUNAKAN XGBOOST REGRESSION (TUNED FINAL):")
display(df_tabel_xgb)

HASIL PREDIKSI MENGGUNAKAN XGBOOST REGRESSION (TUNED FINAL):


,Training (6058),Testing (1515)
R2,0.777,0.831
Mean absolute error,0.232,0.223
Mean squared error,0.145,0.112
Root mean squared error,0.381,0.335
Adjusted R2,0.768,0.799


### Menyimpan Hasil XGBoost

In [36]:
# Simpan Pipeline Model XGBoost Terbaik
joblib.dump(best_model, '../output/house_price_xgb_model.pkl')
print("✅ Model berhasil disimpan: ../output/house_price_xgb_model.pkl")

# Konversi kembali dari skala logaritmik ke skala Rupiah asli untuk keperluan visualisasi grafik
y_test_rupiah = np.expm1(y_test)
y_pred_rupiah = np.expm1(y_pred_test_xgb)

# Simpan Hasil Evaluasi & Prediksi
hasil_xgb = {
    'Model': 'XGBoost + GridSearchCV',
    'Best_Params': best_params,
    'Training': metrik_train_xgb,
    'Testing': metrik_test_xgb,
    'y_test': y_test_rupiah,
    'y_pred_test': y_pred_rupiah
}

joblib.dump(hasil_xgb, '../output/hasil_xgb.pkl')
print("✅ Hasil evaluasi XGBoost berhasil disimpan: ../output/hasil_xgb.pkl")

✅ Model berhasil disimpan: ../output/house_price_xgb_model.pkl
✅ Hasil evaluasi XGBoost berhasil disimpan: ../output/hasil_xgb.pkl
